# More Than A Hack 2027 – Ballinteraktionen untersuchen

**Frage:** Helfen Ballnähe und Änderungen der Ballbewegung dabei, mögliche Spieler-Ball-Interaktionen zu erkennen?

Wir untersuchen den öffentlichen Fünf-Minuten-Bereich als **explorative Videosichtung**, ohne dessen Ground-Truth-Labels einzulesen. Aus zusammenhängend beobachteten Ballpositionen werden geglättete Bewegungen, Beschleunigungspeaks und Abstände zum nächsten Spieler berechnet. Sechs gezielt ausgewählte Szenen stehen mit Signalplots, Einzelbildern und eingebetteten Videos nebeneinander.

**Keine Besitzklassifikation:** Ein Kontaktkandidat ist weder ein bestätigter Kontakt noch bestätigte Ballkontrolle. Wir verändern keine Baseline-Vorhersagen, erzeugen keine H/A/D-Labels, trainieren kein Modell und berichten keine neuen Evaluationsscores.

**Methodische Grundlage:** Link & Hoernig (2017), [Individual ball possession in soccer](https://doi.org/10.1371/journal.pone.0179953), Abschnitte 3.1–3.4. Dieses Notebook ist eine **Teiladaption zur Diagnose**, keine exakte Reproduktion des Papers. Das Paper glättet auch Spielertrajektorien und lernt Distanzschwellen sowie einen Kontrollklassifikator. Hier glätten wir nur den Ball, verwenden unveränderte Spielerpositionen und ausdrücklich unkalibrierte Diagnosewerte.

**Ausführung:** Eigenständiger Python-Kernel mit `numpy`, `pandas`, `matplotlib`, `opencv-python`, `IPython` und lokalem `ffmpeg` → **Restart Kernel → Run All**. Es werden keine Pakete installiert und keine Daten heruntergeladen. Die vollständige Logik bleibt im Notebook. Videoclips werden in einem temporären Ordner erzeugt, in die Ausgaben eingebettet und anschließend von der Festplatte entfernt.

**Nur lokal:** Die Ausgaben enthalten lizenzierte Videobilder und Trackinginformationen. Notebook-Ausgaben vor einem öffentlichen Commit entfernen; die Matchdaten nicht veröffentlichen.

In [ ]:
# Standardbibliothek
from pathlib import Path
from tempfile import TemporaryDirectory
from importlib.metadata import version
import shutil
import subprocess
import sys
import xml.etree.ElementTree as ET

# Daten und Video
import numpy as np
import pandas as pd
import cv2

# Darstellung
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Video

## 1. Alle Annahmen an einer Stelle

Wir benutzen dieselbe metrische Spielfeldgröße wie die offizielle Baseline. Der Untersuchungsbereich ist vorgegeben; die GT wird nicht zur Szenenauswahl oder Parameterbestimmung verwendet.

**Diagnosewerte, keine gemessene Trackinggenauigkeit:** Der RTS-Glätter verwendet ein lineares Positions-/Geschwindigkeitsmodell. Messrauschen σ = 0,25 m und Prozessrauschen σ = 20 m/s² sind feste, illustrative Modellannahmen dieses Notebooks, nicht Werte aus dem Paper. Sie wurden nicht optimiert. Das Prozessmodell nimmt pro Zeitschritt eine zufällige konstante Beschleunigung an; es ist kein validiertes Modell der Flugbahn oder eines Fußkontakts.

Die **4 m/s²** stammen als Beschleunigungshinweis aus dem Paper. **2 m Nähe** sind eine hier gesetzte illustrative Distanzgrenze, kein aus dem Paper übernommener trainierter Wert. Alle Ergebnisse hängen von diesen Annahmen ab. Es findet kein Schwellenvergleich statt.

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "tactical/BAR-ATH_tactical_atd.xml").is_file())
META_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml"
RAW_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt"
VIDEO_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical.mp4"
HOMOGRAPHY_PATH = PROJECT_DIR / "tactical/BAR-ATH_tactical_homography.csv"
FFMPEG = shutil.which("ffmpeg")
FIRST_FRAME, LAST_FRAME = 11185, 18684
PITCH = np.array([105.3, 68.0])
MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2 = 0.25, 20.0
NEAR_DISTANCE_M, PEAK_ACCELERATION_M_S2 = 2.0, 4.0
MIN_RUN_SECONDS, EDGE_GUARD_SECONDS = 1.0, 0.4
CLIP_HALF_SECONDS = 3.0
TEAM_COLORS = {"H": "#1764ab", "A": "#c83e41"}
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False})

if FFMPEG is None:
    raise RuntimeError("Für die eingebetteten Videos wird eine lokale ffmpeg-Installation benötigt.")
print(f"Python {sys.version.split()[0]} | ffmpeg: {FFMPEG}")
display(pd.Series({name: version(name) for name in ["numpy", "pandas", "matplotlib", "opencv-python", "IPython"]}, name="Version").to_frame())
print("Keine Zufallsoperationen, keine Parametersuche, keine Ground-Truth-Labels als Eingabe.")

## 2. Positionsdaten und Zeitachse laden

Wir lesen nur den festgelegten Videoabschnitt aus dem Rohtracking. Die XML bestimmt die Reihenfolge der Track-Slots. Der dritte Rohkanal wird nicht benutzt: Seine Einheit war in den bisherigen Unterlagen widersprüchlich. Geschwindigkeiten werden hier aus den Positionsdaten modelliert, nicht aus diesem Kanal übernommen.

`frame_id − 1` ist der Videoindex. Die Indexprüfung verhindert einen stillen Versatz zwischen Tracking, Videobildern und Signalplots. Alle Frames des Abschnitts bleiben erhalten; nicht beobachtete Positionen bleiben `NaN`.

In [ ]:
root = ET.parse(META_PATH).getroot()
fps = float(root.findtext("./Metadata/GlobalConfig/FrameRate"))
team_nodes = root.findall("./Metadata/Teams/Team")
teams = {t.get("id"): ("H" if i == 0 else "A", t.findtext("Name")) for i, t in enumerate(team_nodes)}
player_nodes = root.findall("./Metadata/Players/Player")
channels = {c.get("id"): c.get("playerId") for c in root.findall("./Metadata/PlayerChannels/PlayerChannel")}
spec = root.find("./DataFormatSpecifications/DataFormatSpecification")
track_ids = [channels[g.find("PlayerChannelRef").get("playerChannelId")]
             for g in spec.findall("./SplitRegister/SplitRegister")]
tracks = pd.DataFrame([{"track_id": p.get("id"), "side": teams[p.get("teamId")][0],
                        "team": teams[p.get("teamId")][1]} for p in player_nodes]).set_index("track_id").loc[track_ids]

frame_ids, positions = [], []
with RAW_PATH.open() as handle:
    for line in handle:
        raw_frame, players_raw, ball_raw = line.strip().split(":")
        frame = int(raw_frame) - 1
        if frame < FIRST_FRAME:
            continue
        if frame > LAST_FRAME:
            break
        frame_ids.append(frame)
        positions.append(np.fromstring(players_raw.replace(";", ",") + "," + ball_raw, sep=",").reshape(-1, 3)[:, :2])
frame_ids, positions = np.array(frame_ids), np.array(positions)
assert np.array_equal(frame_ids, np.arange(FIRST_FRAME, LAST_FRAME + 1)), "Der Trackingabschnitt ist nicht lückenlos framegleich."
player_xy, ball_xy = positions[:, :-1] * PITCH, positions[:, -1] * PITCH
ball_seen = np.isfinite(ball_xy).all(axis=1)
homography = pd.read_csv(HOMOGRAPHY_PATH).set_index("frame")
h_columns = [f"h{i}{j}" for i in range(3) for j in range(3)]

cap = cv2.VideoCapture(str(VIDEO_PATH))
if not cap.isOpened():
    raise RuntimeError(f"Video nicht lesbar: {VIDEO_PATH}")
video_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
assert np.isclose(video_fps, fps) and video_frames == int(spec.get("endFrame")), "Video und Tracking passen zeitlich nicht zusammen."
assert np.array_equal(homography.loc[FIRST_FRAME:LAST_FRAME].index, frame_ids), "Homographien sind nicht framegleich."
print(f"{len(frame_ids):,} Frames bei {fps:g} fps; Ball beobachtet in {100 * ball_seen.mean():.1f} % des Abschnitts.")
display(tracks.groupby(["side", "team"]).size().rename("Track-Slots").to_frame())

## 3. Lückenlose Ballabschnitte getrennt behandeln

Ein Glätter darf keine fehlende Flugbahn durch eine lange Balllücke erfinden. Jeder lückenlos beobachtete Abschnitt wird deshalb separat verarbeitet. Abschnitte unter einer Sekunde erhalten keine Bewegungsdiagnose. Die äußeren 0,4 Sekunden jedes verarbeiteten Abschnitts werden für die Peak-Auswahl gesperrt, weil Ableitungen und Glättung dort besonders von Randannahmen abhängen. Die Rohdaten und ihre Zeilen werden dabei nicht entfernt.

In [ ]:
starts = np.flatnonzero(ball_seen & ~np.r_[False, ball_seen[:-1]])
ends = np.flatnonzero(ball_seen & ~np.r_[ball_seen[1:], False])
observed_runs = pd.DataFrame({"start_offset": starts, "end_offset": ends,
                              "start_frame": frame_ids[starts], "end_frame": frame_ids[ends],
                              "duration_s": (ends - starts + 1) / fps})
observed_runs["processed"] = observed_runs["duration_s"].ge(MIN_RUN_SECONDS)
display(observed_runs[["start_frame", "end_frame", "duration_s", "processed"]].round(2))

## 4. Ballbewegung glätten: kleiner RTS-Glätter im Notebook

Der Zustand besteht aus `(x, y, vx, vy)`. Ein Kalman-Vorwärtsschritt kombiniert Messungen und Bewegungsmodell; der Rauch–Tung–Striebel-Rückwärtsschritt verwendet zusätzlich spätere Beobachtungen **innerhalb desselben Abschnitts**. Das ist eine Offline-Diagnose.

Die Funktion gibt geglättete Positionen und modellierte Geschwindigkeiten zurück. Die Beschleunigung wird anschließend als zeitliche Änderung des zweidimensionalen Geschwindigkeitsvektors berechnet. Damit werden auch Richtungswechsel erfasst, bei denen der Geschwindigkeitsbetrag ähnlich bleibt. Die Funktion bewertet keine Ballkontrolle.

In [ ]:
def smooth_ball_rts(points, fps, measurement_sigma, process_sigma):
    dt = 1 / fps
    transition = np.eye(4)
    transition[0, 2] = transition[1, 3] = dt
    observation = np.eye(2, 4)
    acceleration_map = np.r_[np.eye(2) * dt**2 / 2, np.eye(2) * dt]
    process_noise = acceleration_map @ acceleration_map.T * process_sigma**2
    measurement_noise = np.eye(2) * measurement_sigma**2
    state = np.r_[points[0], (points[1] - points[0]) / dt]
    covariance = np.diag([measurement_sigma**2, measurement_sigma**2, 100.0, 100.0])
    predicted = np.empty((len(points), 4))
    filtered = np.empty_like(predicted)
    predicted_cov = np.empty((len(points), 4, 4))
    filtered_cov = np.empty_like(predicted_cov)

    for i, point in enumerate(points):
        if i:
            state = transition @ state
            covariance = transition @ covariance @ transition.T + process_noise
        predicted[i], predicted_cov[i] = state, covariance
        gain = np.linalg.solve(observation @ covariance @ observation.T + measurement_noise,
                               (covariance @ observation.T).T).T
        state = state + gain @ (point - observation @ state)
        residual_map = np.eye(4) - gain @ observation
        covariance = residual_map @ covariance @ residual_map.T + gain @ measurement_noise @ gain.T
        filtered[i], filtered_cov[i] = state, covariance

    smoothed = filtered.copy()
    for i in range(len(points) - 2, -1, -1):
        gain = np.linalg.solve(predicted_cov[i + 1].T, (filtered_cov[i] @ transition.T).T).T
        smoothed[i] += gain @ (smoothed[i + 1] - predicted[i + 1])
    return smoothed

In [ ]:
signals = pd.DataFrame(index=pd.Index(frame_ids, name="frame"))
signals["ball_observed"] = ball_seen
numeric_columns = ["smooth_x", "smooth_y", "speed_m_s", "acceleration_m_s2", "raw_speed_m_s",
                   "raw_acceleration_m_s2", "nearest_distance_m", "nearest_offset", "smoothing_shift_m"]
signals[numeric_columns] = np.nan
signals["eligible"] = False
edge_guard = round(EDGE_GUARD_SECONDS * fps)

for run in observed_runs.loc[observed_runs["processed"]].itertuples():
    offsets = np.arange(run.start_offset, run.end_offset + 1)
    idx = frame_ids[offsets]
    state = smooth_ball_rts(ball_xy[offsets], fps, MEASUREMENT_SIGMA_M, PROCESS_SIGMA_M_S2)
    raw_velocity = np.gradient(ball_xy[offsets], 1 / fps, axis=0)
    acceleration = np.gradient(state[:, 2:], 1 / fps, axis=0)
    distances = np.linalg.norm(player_xy[offsets] - state[:, None, :2], axis=2)
    distances = np.where(np.isfinite(distances), distances, np.inf)
    nearest = distances.argmin(axis=1)
    minimum = distances.min(axis=1)
    minimum[~np.isfinite(minimum)] = np.nan
    nearest = np.where(np.isfinite(minimum), nearest, np.nan)
    signals.loc[idx, numeric_columns] = np.c_[
        state[:, :2], np.linalg.norm(state[:, 2:], axis=1), np.linalg.norm(acceleration, axis=1),
        np.linalg.norm(raw_velocity, axis=1), np.linalg.norm(np.gradient(raw_velocity, 1 / fps, axis=0), axis=1),
        minimum, nearest, np.linalg.norm(state[:, :2] - ball_xy[offsets], axis=1)]
    if len(idx) > 2 * edge_guard:
        signals.loc[idx[edge_guard:-edge_guard], "eligible"] = True

assert signals.loc[~ball_seen, numeric_columns].isna().all().all(), "Bewegungsmerkmale dürfen keine Balllücken überbrücken."
peak = signals["acceleration_m_s2"].gt(signals["acceleration_m_s2"].shift()) & signals["acceleration_m_s2"].ge(signals["acceleration_m_s2"].shift(-1))
signals["peak"] = peak & signals["eligible"] & signals["acceleration_m_s2"].ge(PEAK_ACCELERATION_M_S2)
signals["near"] = signals["eligible"] & signals["nearest_distance_m"].le(NEAR_DISTANCE_M)
signals["interaction_candidate"] = signals["peak"] & signals["near"]
print(f"Für {signals['eligible'].sum():,} von {len(signals):,} Frames sind Bewegungsmerkmale einschließlich Randabstand auswertbar.")
display(signals.loc[signals["eligible"], ["speed_m_s", "acceleration_m_s2", "nearest_distance_m", "smoothing_shift_m"]].describe(percentiles=[0.5, 0.9, 0.99]).round(3))

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 5), constrained_layout=True)
time_s = (frame_ids - FIRST_FRAME) / fps
axes[0].fill_between(time_s, 0, ball_seen.astype(int), color="#b7cfe1", step="mid", label="Ball beobachtet")
axes[0].plot(time_s, signals["eligible"].astype(int), color="#1764ab", linewidth=1, label="Für Peakdiagnose auswertbar")
axes[0].set(title="Wo sind Ballbewegungen überhaupt auswertbar?", xlabel="Sekunden seit Beginn des öffentlichen Fensters", yticks=[0, 1])
axes[0].legend(loc="lower right")
axes[1].hist(signals.loc[signals["eligible"], "smoothing_shift_m"], bins=35, color="#1764ab")
axes[1].set(title="Wie stark verändert der Glätter die beobachtete Ballposition?", xlabel="Abstand geglättet zu roh (m)", ylabel="Frames")
plt.show()
display(Markdown(f"**Beobachtung:** Der mediane Glättungsversatz beträgt **{signals.loc[signals['eligible'], 'smoothing_shift_m'].median():.3f} m**. Das ist eine Änderung gegenüber dem Eingangssignal, keine gemessene Positionsgenauigkeit. Balllücken bleiben im Merkmalsraum leer."))

## 5. Nähe und Beschleunigungspeaks getrennt betrachten

Die Kandidatenregel lautet: **lokales Maximum der geglätteten Beschleunigung ≥ 4 m/s² und nächster beobachteter Spieler ≤ 2 m entfernt**. Der Abstand bezieht sich auf die geglättete Ballposition und die rohe Spielerposition im selben Frame.

Ein hohes Signal ohne Nähe kann beispielsweise zu einem Ballflug oder zu Trackingfehlern passen. Nähe ohne Peak kann eine ruhige Ballführung, eine weiche Annahme oder fehlerhaftes Balltracking bedeuten. Diese Gruppen sind **keine gelabelten Positiv-/Negativklassen**. Insbesondere kann Ballkontrolle auch ohne einen Peak bestehen.

In [ ]:
diagnostic_counts = pd.Series({
    "Lokale Peaks über Beschleunigungsgrenze": int(signals["peak"].sum()),
    "Davon mit Spieler innerhalb der Nähegrenze": int(signals["interaction_candidate"].sum()),
    "Davon ohne Spieler innerhalb der Nähegrenze": int((signals["peak"] & ~signals["near"]).sum()),
}, name="Anzahl lokaler Signalpeaks")
display(diagnostic_counts.to_frame())
print("Diese Anzahlen messen weder Precision noch Recall und sind keine Anzahl echter Kontakte.")

## 6. Szenenauswahl und Grenzen der Sichtprüfung

Die folgenden sechs Frames wurden **explorativ aus der Signalübersicht und Videosichtung ausgewählt**, um unterschiedliche Konstellationen zu zeigen. Die Auswahl ist gezielt, nicht zufällig oder repräsentativ. Die Frame-IDs sind für eine identische Wiederholung festgehalten; sie verändern keine Vorhersage.

Die Beschreibungen sind vorsichtige Sichtungsnotizen, keine Ground Truth. Insbesondere wurden kein eindeutig isolierter Abpraller und keine zweifelsfrei direkte Ein-Kontakt-Weiterleitung als eigene Klasse bestätigt. Diese Szenentypen werden deshalb nicht als erfolgreich erkannt behauptet.

Die Videos enthalten ±3 Sekunden Kontext. Die Standbilder zeigen ±1 Sekunde in einem festen Ausschnitt. Gelbe Kreise markieren die Projektion der **rohen Ballposition auf die Spielfeldebene**; sie sind keine zusätzliche Video-Balldetektion. Bei hohen Bällen kann der echte Ball oberhalb der Markierung liegen. Auch ein falscher Balltrack kann eine scheinbar plausible Markierung erzeugen.

In [ ]:
scenes = pd.DataFrame([
    {"frame": 12217, "scene": "Passannahme und anschließende Aktion", "note": "Ein Pass erreicht einen Barcelona-Spieler; danach wird der Ball weitergespielt. Der Peak ist ein plausibler Interaktionshinweis, bestätigt aber nicht die genaue Kontaktzahl."},
    {"frame": 14516, "scene": "Kurze Ballaktion mit Gegnernähe", "note": "Ein Barcelona-Spieler erhält den Ball und spielt ihn im Nahbereich eines Gegners weiter. Ob dies exakt eine direkte Ein-Kontakt-Aktion ist, bleibt ohne feinere Sichtung offen."},
    {"frame": 15686, "scene": "Torwartabspiel und Ballflug", "note": "Im Videokontext ist ein Torwartabspiel zu sehen. Im anschließenden Ballflug weicht die projizierte Bodenposition sichtbar von der Bildposition des Balls ab; das ist keine verlässliche Messung der Ballhöhe."},
    {"frame": 12676, "scene": "Hoher Ball und Annäherung", "note": "Der Ball ist im Video in der Luft zu sehen. Ein Beschleunigungspeak liegt vor, aber der nächste beobachtete Spieler ist an diesem Frame außerhalb der 2-m-Diagnosegrenze."},
    {"frame": 13705, "scene": "Starker Peak im Passgeschehen", "note": "Die Sequenz enthält eine Ballweiterleitung mit deutlicher Bewegungsänderung. Der große Peak allein beantwortet nicht, ob ein Team kontrollierten Besitz gewinnt oder behält."},
    {"frame": 11639, "scene": "Nähe bei schwacher gemessener Beschleunigung", "note": "Im Torraum ist eine dynamische Spielszene zu sehen, während das gemessene Ballsignal nur langsam läuft. Die Markierung muss kritisch mit dem sichtbaren Ball verglichen werden; Nähe allein bestätigt hier keine Kontrolle."},
]).set_index("frame")
scene_summary = scenes[["scene"]].join(signals[["nearest_distance_m", "acceleration_m_s2", "speed_m_s", "interaction_candidate"]])
scene_summary["video_s"] = scene_summary.index / fps
display(scene_summary.round(2))

## 7. Gemeinsame Darstellung für jede Szene

Die Signalplots zeigen Roh- und geglättete Bewegung sowie den Abstand zum nächsten beobachteten Track. Sie verwenden dieselbe Zeitachse wie das Video. Auf der rechten y-Achse des letzten Plots steht der Betrag des Glättungsversatzes.

Die Anfangsgeschwindigkeitsvarianz des Glätters ist fest auf 100 (m/s)² gesetzt; sie beschreibt nur die unsichere Initialisierung. Die eingebetteten Ausschnitte werden mit `ffmpeg` aus dem lokalen Original erzeugt. Das ist reine Darstellung, kein weiterer Detektor. In Oberflächen ohne Videowiedergabe bleiben die Standbilder und Signalplots verfügbar.

In [ ]:
def show_scene(frame):
    row = scenes.loc[frame]
    display(Markdown(f"### Frame {frame}: {row['scene']}\n\nVideosekunde **{frame / fps:.2f}**. {row['note']}"))
    window = signals.loc[max(FIRST_FRAME, frame - round(CLIP_HALF_SECONDS * fps)):
                         min(LAST_FRAME, frame + round(CLIP_HALF_SECONDS * fps))]
    relative_s = (window.index.to_numpy() - frame) / fps
    fig, axes = plt.subplots(3, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
    axes[0].plot(relative_s, window["raw_speed_m_s"], color="0.65", label="Aus Rohpositionen differenziert")
    axes[0].plot(relative_s, window["speed_m_s"], color="#1764ab", label="RTS-Geschwindigkeit")
    axes[0].set(ylabel="Geschwindigkeit (m/s)", title=f"Bewegungsdiagnose – Frame {frame}")
    axes[0].legend(loc="upper right")
    axes[1].plot(relative_s, window["raw_acceleration_m_s2"], color="0.65", label="Aus Rohpositionen differenziert")
    axes[1].plot(relative_s, window["acceleration_m_s2"], color="#1764ab", label="Aus RTS-Geschwindigkeit")
    axes[1].axhline(PEAK_ACCELERATION_M_S2, color="#c83e41", linestyle="--", label="4 m/s² (Paper-Hinweis)")
    marked = window["interaction_candidate"]
    axes[1].scatter(relative_s[marked], window.loc[marked, "acceleration_m_s2"], color="#c83e41", s=20, zorder=3)
    axes[1].set(ylabel="Beschleunigung (m/s²)")
    axes[1].legend(loc="upper right", fontsize=8)
    axes[2].plot(relative_s, window["nearest_distance_m"], color="#1764ab", label="Abstand zum nächsten Track")
    axes[2].axhline(NEAR_DISTANCE_M, color="#c83e41", linestyle="--", label="2 m (illustrativ)")
    axes[2].set(ylabel="Spieler–Ball (m)", xlabel="Sekunden relativ zum ausgewählten Frame")
    axes[2].legend(loc="upper left", fontsize=8)
    shift_axis = axes[2].twinx()
    shift_axis.plot(relative_s, window["smoothing_shift_m"], color="#a66c00", alpha=0.7)
    shift_axis.set_ylabel("Glättungsversatz (m)", color="#a66c00")
    for ax in axes:
        ax.axvline(0, color="black", linewidth=1, alpha=0.6)
        ax.grid(alpha=0.2)
    plt.show()

    capture = cv2.VideoCapture(str(VIDEO_PATH))
    if homography.loc[frame, "valid"] != 1:
        capture.release()
        raise ValueError(f"Homographie für Frame {frame} nicht gültig.")
    center_point = np.r_[positions[frame - FIRST_FRAME, -1], 1.0] @ homography.loc[frame, h_columns].to_numpy().reshape(3, 3).T
    center_point = center_point[:2] / center_point[2]
    width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    crop_width, crop_height = min(800, width), min(500, height)
    left = int(np.clip(center_point[0] - crop_width / 2, 0, width - crop_width))
    top = int(np.clip(center_point[1] - crop_height / 2, 0, height - crop_height))
    fig, axes = plt.subplots(2, 3, figsize=(14, 6.2), constrained_layout=True)
    for ax, delta in zip(axes.ravel(), [-25, -15, 0, 5, 15, 25]):
        target = frame + delta
        capture.set(cv2.CAP_PROP_POS_FRAMES, target)
        ok, image = capture.read()
        if not ok:
            capture.release()
            raise RuntimeError(f"Video-Frame {target} nicht lesbar.")
        ax.imshow(cv2.cvtColor(image[top:top + crop_height, left:left + crop_width], cv2.COLOR_BGR2RGB))
        if FIRST_FRAME <= target <= LAST_FRAME and homography.loc[target, "valid"] == 1:
            point = np.r_[positions[target - FIRST_FRAME, -1], 1.0] @ homography.loc[target, h_columns].to_numpy().reshape(3, 3).T
            if np.isfinite(point).all() and abs(point[2]) > 1e-10:
                pixel = point[:2] / point[2]
                ax.scatter(pixel[0] - left, pixel[1] - top, s=120, facecolors="none", edgecolors="#ffdf00", linewidths=1.2)
        ax.set(xlim=(0, crop_width), ylim=(crop_height, 0), title=f"Frame {target} | {delta / fps:+.1f} s")
        ax.axis("off")
    capture.release()
    fig.suptitle("Fester Videoausschnitt – gelb: projizierte rohe Ballposition", fontsize=11)
    plt.show()

    with TemporaryDirectory(prefix="mtah03-video-") as temporary:
        clip = Path(temporary) / "scene.mp4"
        subprocess.run([FFMPEG, "-hide_banner", "-loglevel", "error", "-ss", str(frame / fps - CLIP_HALF_SECONDS),
                        "-i", str(VIDEO_PATH), "-t", str(2 * CLIP_HALF_SECONDS), "-an", "-vf", "scale=1280:-2",
                        "-c:v", "libx264", "-preset", "veryfast", "-crf", "25", "-pix_fmt", "yuv420p",
                        "-movflags", "+faststart", str(clip)], capture_output=True, text=True, check=True)
        display(Video(data=clip.read_bytes(), embed=True, mimetype="video/mp4", width=900,
                      html_attributes='controls preload="none"'))
    candidate = signals.loc[frame]
    nearest_offset = candidate["nearest_offset"]
    nearest_track = track_ids[int(nearest_offset)] if np.isfinite(nearest_offset) else "keiner"
    display(Markdown(f"**Signal am ausgewählten Frame:** Abstand **{candidate['nearest_distance_m']:.2f} m**, "
                     f"Beschleunigung **{candidate['acceleration_m_s2']:.2f} m/s²**, nächster Track **{nearest_track}** "
                     f"(keine dauerhafte Spieler-ID). Kandidatenregel erfüllt: **{bool(candidate['interaction_candidate'])}**. "
                     "Dieser Status ist eine technische Diagnose, kein manuelles Kontakt- oder Besitzlabel."))

In [ ]:
show_scene(12217)

In [ ]:
show_scene(14516)

In [ ]:
show_scene(15686)

In [ ]:
show_scene(12676)

In [ ]:
show_scene(13705)

In [ ]:
show_scene(11639)

## 8. Aktueller Befund und offene Fragen

**Beobachtung aus der Sichtung:** In den gezeigten Annahme-/Abspielszenen liegen plausible Interaktionshinweise vor. Die Flugballszenen zeigen zugleich, dass die Projektion auf die Spielfeldebene und die sichtbare Ballposition auseinanderliegen können. Die Vergleichsszene mit schwacher gemessener Beschleunigung zeigt, warum die Verlässlichkeit des Balltracks separat geprüft werden muss.

**Interpretation:** Ballnähe und Beschleunigung können beim Auffinden interessanter Momente helfen. Aus diesen sechs gezielt ausgewählten Beispielen folgt keine gemessene Erkennungsrate. Ein Peak bestätigt weder die Art des Kontakts noch kontrollierten Team-Ballbesitz. Umgekehrt widerlegt ein fehlender Peak keine Kontrolle. Fehlende Track-Slots, Ballhöhe, Verdeckungen, Zeitversatz und Trackingfehler bleiben mögliche Ursachen irreführender Abstände.

**Grenze der Übertragung:** Das Paper setzt ein Ball-im-Spiel-Signal voraus und verwendet zusätzlich trainierte Kontrollklassifikation. Beides wurde hier nicht ergänzt. Die Unterbrechungserkennung unserer Baseline bleibt deshalb unverändert.

In [ ]:
display(Markdown(
    f"- **Datenbereich:** {len(frame_ids):,} Frames, davon **{signals['eligible'].sum():,}** für die Peakdiagnose auswertbar.\n"
    f"- **Technische Kandidaten:** **{signals['interaction_candidate'].sum()}** lokale Peaks mit Nähe unter den dokumentierten Annahmen.\n"
    f"- **Sichtung:** **{len(scenes)}** gezielt ausgewählte Szenen mit Signalplots, Standbildern und Videos.\n"
    "- **Keine Leistungsbewertung:** Keine Kontakt-Ground-Truth, keine Precision/Recall-Werte und kein neuer spellF1.\n"
    "- **Bestehende Ergebnisse:** Die Vorhersagen und Evaluationsdateien aus Notebook 02 wurden nicht verändert."
))

**Nächste Entscheidung:**

A) Eine kleine, getrennte Stichprobe visuell annotieren, um Kontaktkandidaten und tatsächliche Kontrolle auseinanderzuhalten.

B) Zuerst die Ball-im-Spiel-/Unterbrechungserkennung untersuchen, da sie im aktuellen Baselinevergleich besonders schwach war.

Für die Beurteilung der hier untersuchten Interaktionsidee wäre A methodisch sinnvoll: Ohne unabhängige Sichtungslabels können wir einen kontaktähnlichen Peak nicht als Treffer zählen. Die Stichprobe müsste auch echte Kontakte ohne Peak enthalten, sonst wäre kein Recall bestimmbar. Es wurden noch keine solchen Labels erstellt und keine Parameter anhand der Szenen angepasst.